# SAR Oil Spill Detection – U-Net

U-Net for oil spill segmentation on **Zenodo Sentinel-1 SAR** TIFF data.

**Dataset:** `oil_spill_Final/` with train/val/test splits (TIFF format).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========== CONFIG ==========
# Suppress rasterio/GDAL TIFF warnings (SAR TIFFs have nonstandard photometric tags)
import os
os.environ['CPL_LOG'] = 'NUL' if os.name == 'nt' else '/dev/null'
import sys
import warnings
warnings.filterwarnings('ignore', message='.*Photometric type-related.*')
warnings.filterwarnings('ignore', module='rasterio')
import logging
for _ in ['rasterio', 'rasterio._env']:
    logging.getLogger(_).setLevel(logging.ERROR)

# Path to reorganized SAR dataset (train/val/test)
if os.path.exists("/content/drive"):
    BASE = "/content/drive/MyDrive/oil_spill_Final"
else:
    BASE = r"C:\Users\Akshat\OneDrive\Documents\DeepSpill\oil_spill_Final"

IMG_HEIGHT, IMG_WIDTH = 256, 256
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 1)

# Add scripts to path for load_zenodo_sar
_scripts = os.path.join(os.getcwd(), "scripts")
if os.path.isdir(_scripts) and _scripts not in sys.path:
    sys.path.insert(0, _scripts)

print("Dataset base:", BASE)

In [ ]:
# ========== LOAD SAR DATA (TIFF) ==========
import numpy as np
from load_zenodo_sar import load_from_simple_structure

print("Loading train...")
X_train, Y_train = load_from_simple_structure(
    base_dir=BASE,
    split='train',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    channels=1,
    normalize_sar=True,
)
print("Loading val...")
X_val, Y_val = load_from_simple_structure(
    base_dir=BASE,
    split='val',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    channels=1,
    normalize_sar=True,
)
print("Loading test...")
X_test, Y_test = load_from_simple_structure(
    base_dir=BASE,
    split='test',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    channels=1,
    normalize_sar=True,
)

print(f"Train: {X_train.shape} {Y_train.shape}")
print(f"Val:   {X_val.shape} {Y_val.shape}")
print(f"Test:  {X_test.shape} {Y_test.shape}")

In [ ]:
# ========== U-NET MODEL ==========
import tensorflow as tf
from tensorflow.keras import layers, models

def conv_block(x, filters, kernel_size=3, activation='relu', padding='same', use_bn=True, dropout=0.0):
    x = layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    x = layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

def encoder_block(x, filters, dropout=0.0):
    c = conv_block(x, filters, dropout=dropout)
    p = layers.MaxPooling2D((2,2))(c)
    return c, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, (2,2), strides=(2,2), padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256,256,1), num_classes=1, base_filters=32, dropout_rate=0.1):
    inputs = layers.Input(shape=input_shape)
    c1, p1 = encoder_block(inputs, base_filters, dropout=dropout_rate)
    c2, p2 = encoder_block(p1, base_filters*2, dropout=dropout_rate)
    c3, p3 = encoder_block(p2, base_filters*4, dropout=dropout_rate)
    c4, p4 = encoder_block(p3, base_filters*8, dropout=dropout_rate)
    b = conv_block(p4, base_filters*16, dropout=dropout_rate)
    d4 = decoder_block(b, c4, base_filters*8)
    d3 = decoder_block(d4, c3, base_filters*4)
    d2 = decoder_block(d3, c2, base_filters*2)
    d1 = decoder_block(d2, c1, base_filters)
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid', dtype='float32')(d1)
    return models.Model(inputs, outputs)

unet = build_unet(input_shape=INPUT_SHAPE, num_classes=1, base_filters=32, dropout_rate=0.1)
unet.summary()

In [ ]:
# ========== METRICS & LOSS ==========
import tensorflow.keras.backend as K

def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = K.batch_flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.batch_flatten(tf.cast(y_pred, tf.float32))
    inter = K.sum(y_true_f * y_pred_f, axis=-1)
    dice = (2.*inter + smooth) / (K.sum(y_true_f, axis=-1) + K.sum(y_pred_f, axis=-1) + smooth)
    return K.mean(dice)

def iou_metric(y_true, y_pred, smooth=1):
    y_true_f = K.batch_flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.batch_flatten(tf.cast(y_pred, tf.float32))
    inter = K.sum(y_true_f * y_pred_f, axis=-1)
    union = K.sum(y_true_f, axis=-1) + K.sum(y_pred_f, axis=-1) - inter
    return K.mean((inter + smooth) / (union + smooth))

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce, axis=[1, 2])
    d = dice_loss(y_true, y_pred)
    return tf.reduce_mean(bce + d)

In [ ]:
# ========== AUGMENTATION & DATA PIPELINE ==========
def augment_image_mask_tf(img, mask):
    concat = tf.concat([img, mask], axis=-1)
    concat = tf.image.random_flip_left_right(concat)
    concat = tf.image.random_flip_up_down(concat)
    concat = tf.keras.layers.RandomRotation(0.15)(concat)
    concat = tf.keras.layers.RandomZoom(0.1)(concat)
    img_aug = concat[..., :1]
    mask_aug = tf.cast(concat[..., 1:2] > 0.5, tf.float32)
    img_aug = tf.image.random_brightness(img_aug, max_delta=0.2)
    img_aug = tf.clip_by_value(img_aug, 0.0, 1.0)
    img_aug = tf.image.random_contrast(img_aug, lower=0.8, upper=1.2)
    img_aug = tf.clip_by_value(img_aug, 0.0, 1.0)
    return img_aug, mask_aug

BATCH_SIZE = 8
train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_ds = train_ds.shuffle(500).repeat()
train_ds = train_ds.map(augment_image_mask_tf, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = max(1, len(X_train) // BATCH_SIZE)
validation_steps = max(1, len(X_val) // BATCH_SIZE)
print(f"Steps/epoch: {steps_per_epoch}, Val steps: {validation_steps}")

In [ ]:
# ========== TRAINING ==========
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

unet.compile(
    optimizer='adam',
    loss=combined_loss,
    metrics=['accuracy', iou_metric, dice_coefficient]
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint('unet_sar_oilspill_best.keras', save_best_only=True, monitor='val_loss', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
]

history = unet.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    epochs=80,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ========== EVALUATE ON TEST SET ==========
test_metrics = unet.evaluate(X_test, Y_test)
print(f"Test loss: {test_metrics[0]:.4f}")
print(f"Test accuracy: {test_metrics[1]:.4f}")
print(f"Test IoU: {test_metrics[2]:.4f}")
print(f"Test Dice: {test_metrics[3]:.4f}")

In [ ]:
# ========== VISUALIZE PREDICTIONS ==========
import matplotlib.pyplot as plt

n_show = 4
idx = np.random.choice(len(X_test), min(n_show, len(X_test)), replace=False)
preds = unet.predict(X_test[idx], verbose=0)
preds_bin = (preds > 0.5).astype(np.float32)

fig, axes = plt.subplots(n_show, 3, figsize=(12, 4*n_show))
for i in range(n_show):
    axes[i, 0].imshow(X_test[idx[i]].squeeze(), cmap='gray')
    axes[i, 0].set_title('SAR')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(Y_test[idx[i]].squeeze(), cmap='gray')
    axes[i, 1].set_title('GT')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(preds_bin[i].squeeze(), cmap='gray')
    axes[i, 2].set_title('Pred')
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()